SPDX-FileCopyrightText: © 2026 Idiap Research Institute <contact@idiap.ch>
SPDX-FileContributor: David Alonso del Barrio <ddbarrio@idiap.ch>
SPDX-License-Identifier: GPL-3.0-only

# EMN Factsheet Standard Extraction Pipeline
The notebook below demonstrates the automated extraction pipeline using a small sample of 5 PDFs included in this repository. The emn_22_cleaned.csv file provided in this repo represents the cleaned output for these 5 samples only, result of applying the pipeline below and performing the subsequent manual cleaning.

The full dataset presented in our paper (covering 35 countries and 13 years)is in our Zenodo repository: https://doi.org/10.34777/fxzr-nc56

In [1]:
import os
import fitz
import re
import pandas as pd

folder_path = "./data/2022/sample" 

### Extraction Process:
We show the extardction process from PDFs to tabualr data 

In [2]:
def extract_text_mupdf(pdf_path):
    """
    Parses text content from a PDF file using PyMuPDF.
    
    Args:
        pdf_path (str): File path to the source PDF.
        
    Returns:
        str: Combined text from all pages with preserved line breaks.
    """
    with fitz.open(pdf_path) as doc:
        text = "\n".join([page.get_text("text") for page in doc])
    return text

In [3]:
def format_text_section(text):
    """
    Normalizes raw text extracted from 2022 EMN factsheets.
    
    This function performs multi-stage cleaning to harmonize content:
    1. Extracts core content based on year-specific document markers.
    2. Removes EMN-specific metadata.
    3. Restructures content into Title-Paragraph pairs.
    
    Args:
        text (str): The raw text extracted from the PDF.
        
    Returns:
        str: The structured, cleaned text, or None if extraction fails.
    """
    # Define document boundaries for 2022 format
    match = re.search(r"(KEY POINTS.*?)(?:STATISTICAL ANNEX)", text, re.DOTALL)

    if not match:
        print("Warning: Could not find target section markers (KEY POINTS/STATISTICAL ANNEX).")
        return None

    cleaned_text = match.group(1)

    # 1. Strip unwanted EMN metadata headers
    cleaned_text = re.sub(r"\d+\sEMN\sCountry\sFactsheet:\s[A-Z]+\n?", '', cleaned_text)

    # 2. Reconstruct split section titles
    cleaned_text = re.sub(r"(\b[A-Z]+(?:\s+[A-Z]+)*)(\s*-\s*\n)([A-Z]+(?:\s+[A-Z]+)*)", r"\1 \3", cleaned_text)
    cleaned_text = re.sub(r"(?<!\n)([A-Z][A-Z ]{2,})\b", r"\n\1", cleaned_text)

    # 3. Clean up residual whitespace artifacts
    cleaned_text = re.sub(r"(?<=\n)([A-Z])\s*\n([A-Z])", r"\1\2", cleaned_text)
    cleaned_text = re.sub(r"\n\s+|\s+\n", "\n", cleaned_text)

    # 4. Structure text into Title-Paragraph pairs
    sections = re.split(r"(\n[A-Z ]{2,}\n)", cleaned_text)
    final_text = []

    for section in sections:
        section = section.strip()
        if not section:
            continue
        # Apply formatting logic: titles stay uppercase, content is joined as a paragraph
        if section.isupper():
            final_text.append(section)
        else:
            final_text.append(re.sub(r"\s+", " ", section))

    return "\n\n".join(final_text)

In [4]:
def process_pdfs(folder_path):
    """
    Orchestrates the extraction and formatting pipeline for 2022 factsheets.
    
    Note: As described in our methodology, certain documents may require 
    manual post-processing (e.g., handling missing header markers) if the 
    automated extraction fails to identify key document boundaries.
    
    Args:
        folder_path (str): Directory containing the PDF factsheets.
        
    Returns:
        pd.DataFrame: Structured dataset with 'country', 'year', and 'text' columns.
    """
    data = []
    
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Path not found: {folder_path}")

    for filename in os.listdir(folder_path):
        if not filename.endswith('.pdf'):
            continue
            
        pdf_path = os.path.join(folder_path, filename)
        
        # 1. Automated Extraction
        raw_text = extract_text_mupdf(pdf_path)
        if not raw_text:
            continue
            
        # 2. Automated Formatting
        formatted_text = format_text_section(raw_text)
        
        # 3. Metadata and Storage
        # Note: If extraction returns None, document may require manual review
        if formatted_text:
            parts = filename.split('-')
            if len(parts) >= 4:
                data.append({
                    'country': parts[2],
                    'year': parts[3][:4],
                    'text': formatted_text
                })
        else:
            print(f"Manual review required for: {filename}")
    
    return pd.DataFrame(data)

In [5]:
df = process_pdfs(folder_path)
df
# df.to_csv("extracted_factsheets.csv", index=False)

,country,year,text
0,Bulgaria,2022,KEY POINTS\n\nFollowing the activation of the ...
1,Austria,2022,KEY POINTS\n\nReform of the Red-White-Red Card...
2,Belgium,2022,KEY POINTS\n\nMore than 100 000 people applied...
3,Croatia,2022,"KEY POINTS\n\nOn 8 December 2022, the Council ..."
4,Armenia,2022,"KEY POINTS\n\nOn 27 October 2022, the Head of ..."


### Visualizing Extraction Quality:
Below, we compare a sample of the raw output from our pipeline to the cleaned version. This illustrates the impact of the manual curation phase. This is the pdf of reference for this example: ./data/2022/sample/EMN-factsheet-Bulgaria-2022_en.pdf 

In [6]:
print(df.text[0])

KEY POINTS

Following the activation of the EU Temporary Protection Directive, the Bulgarian government adopted Decision No 144/10.03.2022 to grant temporary protection to displaced persons from Ukraine. In 2022, special focus was placed on Directive (EU) 2021/1883 on the conditions of entry and residence of third-country nationals for the purpose of highly qualified employment, and repealing Council Directive 2009/50/EC. The national parliament (National Assembly) proactively legislated for the partial introduction of the new Blue Card in the Law on Foreigners in the Republic of Bulgaria ( LFRB).

LEGAL MIGRATION AND

MOBILITY Overarching legal or policy changes in legal migration The proposed draft legislation amending the Law on Foreigners in the Republic of Bulgaria ( LFRB) envisages the possibility to obtain a five-year residence-and-work permit for highly qualified employment, including for beneficiaries of international protection. Work-related migration Amendments were introduc

### Note on Data Quality
The raw extraction above produces the initial structure. 
As described in the paper, this dataset was subjected to a three-pass manual review and cleaning process. 
The cleaned version is loaded below, and we show the same example as above, but in this case all cleaned. Reference to evaluate the quality: ./data/2022/sample/EMN-factsheet-Bulgaria-2022_en.pdf

In [7]:
df_22=pd.read_csv('./data/2022/final/emn_22_cleaned.csv')
print(df_22.text[0])

KEY POINTS
Following the activation of the EU Temporary Protection Directive, the Bulgarian government adopted Decision No 144/10.03.2022 to grant temporary protection to displaced persons from Ukraine. In 2022, special focus was placed on Directive (EU) 2021/1883 on the conditions of entry and residence of third-country nationals for the purpose of highly qualified employment, and repealing Council Directive 2009/50/EC. The national parliament (National Assembly) proactively legislated for the partial introduction of the new Blue Card in the Law on Foreigners in the Republic of Bulgaria (LFRB).

LEGAL MIGRATION AND MOBILITY
OVERARCHING LEGAL OR POLICY CHANGES IN LEGAL MIGRATION
The proposed draft legislation amending the Law on Foreigners in the Republic of Bulgaria (LFRB) envisages the possibility to obtain a five-year residence-and-work permit for highly qualified employment, including for beneficiaries of international protection.

WORK-RELATED MIGRATION
Amendments were introduced 